In [120]:
import re
import string
import pandas as pd
import torch 
import datasets as ds
import os
import random
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Define your target device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

# get the data
d = ds.load_dataset("sentence-transformers/natural-questions", split="train", streaming=True)
data = d.shuffle(seed=random.randint(0,20000000), buffer_size=10000).take(1000).to_pandas()
data = data.drop('answer',axis=1) # drops the answers
data['query'] = data['query'].str.replace(r'\W ', '', regex=True) # remove non alphanum chars
print(data)


#word_unique = set(data['query'].str.lower().str.cat(sep=' ').split())
#print(word_unique)
#print(len(word_unique))


                                                 query
0       what is the average weight of a humpback whale
1    where is the ex post facto law in the constitu...
2     who sang these boots were made for walking first
3    samuel de champlain provincial park nipissing ...
4           who said the whole is greater than the sum
..                                                 ...
995  write the name of two kings of chahamanas dynasty
996  which adaptations should you have to enable yo...
997               who sings once i was seven years old
998  when is a return statement required inside a f...
999               who was the queen of england in 1890

[1000 rows x 1 columns]


In [124]:
from collections import Counter
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, args):
        self.args = args
        self.words = self.load_words()
        self.unique_words = self.get_unique_words()

        self.index_to_word = {index: word for index,\
                              word in enumerate(self.unique_words)}
        self.word_to_index = {word: index for index, \
                              word in enumerate(self.unique_words)}

        self.word_indexes = [self.word_to_index[w] for w in self.words]

    def load_words(self):
        text = data['query'].str.cat(sep=' ')
        return text.split(' ')

    def get_unique_words(self):
        word_counts = Counter(self.words)
        return sorted(word_counts, key=word_counts.get, reverse=True)

    def __len__(self):
        return len(self.word_indexes) - self.args

    def __getitem__(self, index):
        return (
            torch.tensor(self.word_indexes[index:index + self.args]),
            torch.tensor(self.word_indexes[index + 1:index + self.args+ 1])
        )

In [125]:
from torch import nn
class LSTMModel(nn.Module):
    def __init__(self, dataset):
        super(LSTMModel, self).__init__()
        self.lstm_size = 128
        self.embedding_dim = 128
        self.num_layers = 3

        n_vocab = len(dataset.unique_words)
        self.embedding = nn.Embedding(
            num_embeddings=n_vocab,
            embedding_dim=self.embedding_dim,
        )
        self.embedding.to(device)
        self.lstm = nn.LSTM(
            input_size=self.embedding_dim,
            hidden_size=self.lstm_size,
            num_layers=self.num_layers,
            dropout=0.2,
        )
        self.lstm.to(device)
        self.fc = nn.Linear(self.lstm_size, n_vocab)
        self.fc.to(device)

    def forward(self, x, prev_state):
        embed = self.embedding(x)
        output, state = self.lstm(embed, prev_state)
        logits = self.fc(output)

        return logits, state

    def init_state(self, sequence_length):
        return (
            torch.zeros(self.num_layers, \
                        sequence_length, self.lstm_size).to(device),
            torch.zeros(self.num_layers, \
                        sequence_length, self.lstm_size).to(device)
        )

In [ ]:
from torch.utils.data import  DataLoader, random_split, Subset

# Hyperparameters
sequence_length = 10
batch_size = 64
learning_rate = 0.001
num_epochs = 10

# Create the dataset
dataset = TextDataset(sequence_length)
# Split the dataset into training and validation sets

total_len = len(dataset)
split_idx = int(0.8 * total_len)
train_indices = list(range(0, split_idx))
val_indices = list(range(split_idx, total_len))

# 3. Create Subsets
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

# Create data loaders
train_loader = DataLoader(train_dataset,
                      batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset,
                        batch_size=batch_size)

# Create the model
model = LSTMModel(dataset).to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),\
                             lr=learning_rate)
# https://www.geeksforgeeks.org/data-science/sentence-autocomplete-using-pytorch/
# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs, targets = batch

        optimizer.zero_grad()

        hidden = model.init_state(sequence_length)
        outputs, _ = model(inputs, hidden)

        loss = criterion(outputs.view(-1,
                      len(dataset.unique_words)), \
                         targets.view(-1))
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    # Calculate average loss for the epoch
    average_loss = total_loss / len(train_loader)

    # Print the epoch and average loss
    print(f"Epoch [{epoch+1}/{num_epochs}],\
                    Average Loss: {average_loss:.4f}")

    # Validation loop
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            inputs, targets = batch

            hidden = model.init_state(sequence_length)
            outputs, _ = model(inputs, hidden)

            loss = criterion(outputs.view(-1,
                              len(dataset.unique_words)), \
                             targets.view(-1))
            val_loss += loss.item()

    # Calculate average validation loss for the epoch
    average_val_loss = val_loss / len(val_loader)

    # Print the epoch and average validation loss
    print(f"Epoch[{epoch+1}/{num_epochs}], Validation Loss: {average_val_loss: .4f}")

RuntimeError: Expected a 'cuda' device type for generator but found 'cpu'